# Phase 3: Feature Extraction & Representation
### Project: Automated Classification of Martian Surface Images Captured by NASA's Curiosity Rover Using Machine Learning

This notebook demonstrates the deterministic feature extraction pipeline that converts standardized `(256, 256, 3)` Martian surface images into compact numerical feature vectors for conventional ML models:
1. **Group A: Color Statistics (19 features)** - Per-channel RGB stats & luminance brightness
2. **Group B: Color Histograms (48 features)** - 16 normalized bins per RGB channel
3. **Group C: GLCM Texture (12 features)** - Contrast, dissimilarity, homogeneity, energy, correlation, ASM
4. **Group D: HOG Features (8,100 features)** - Histogram of Oriented Gradients (9 orientations, 16x16 pixels/cell, 2x2 cells/block)
5. **Group E: Edge Descriptors (2 features)** - Canny edge density & mean edge gradient magnitude
6. **Group F: Gradient Statistics (5 features)** - Sobel magnitude moments

**Total Features per Image**: **8,186 features**

> **Dataset Preservation**: Exact class counts and natural class imbalance (~122x) are 100% preserved without any rebalancing, synthesis, or global data leakage.

In [ ]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is on sys.path
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import config
from preprocessing.image_processing import preprocess_image
from preprocessing.feature_extraction import (
    get_feature_group_breakdown,
    get_feature_names,
    extract_features_from_preprocessed_image,
    extract_image_features
)

print(f"Project root: {config.PROJECT_ROOT}")
print(f"Features directory: {config.FEATURES_DIR}")

## 1. Programmatic Feature Group Breakdown
Verify the programmatic count of features across each individual group.

In [ ]:
breakdown = get_feature_group_breakdown()
df_breakdown = pd.DataFrame(list(breakdown.items()), columns=["Feature Group", "Feature Count"])
df_breakdown

## 2. Extracting & Inspecting Features on a Representative Image
Let's inspect the visual features extracted from a sample Curiosity rover image.

In [ ]:
sample_path = config.CALIBRATED_IMG_DIR / "0077ML0005780000102730I01_DRCL.JPG"
preprocessed = preprocess_image(sample_path)

feat_vec, feat_names = extract_features_from_preprocessed_image(preprocessed, return_names=True)
print(f"Feature vector shape: {feat_vec.shape}, dtype: {feat_vec.dtype}")
print(f"Total feature names : {len(feat_names)}")
print(f"Min value: {feat_vec.min():.4f}, Max value: {feat_vec.max():.4f}")
print(f"Sample first 10 features: {feat_names[:10]}")
print(f"Sample first 10 values  : {feat_vec[:10]}")

## 3. Loading & Validating Saved Feature Artifacts
Load the compressed `.npz` feature matrices and verify split dimensions and finite values.

In [ ]:
X_train = np.load(config.FEATURES_DIR / "X_train.npz")["X"]
y_train = np.load(config.FEATURES_DIR / "y_train.npy")
X_val = np.load(config.FEATURES_DIR / "X_val.npz")["X"]
y_val = np.load(config.FEATURES_DIR / "y_val.npy")
X_test = np.load(config.FEATURES_DIR / "X_test.npz")["X"]
y_test = np.load(config.FEATURES_DIR / "y_test.npy")

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val   shape: {X_val.shape}, y_val   shape: {y_val.shape}")
print(f"X_test  shape: {X_test.shape}, y_test  shape: {y_test.shape}")

assert np.isfinite(X_train).all(), "Non-finite values found in X_train!"
assert np.isfinite(X_val).all(), "Non-finite values found in X_val!"
assert np.isfinite(X_test).all(), "Non-finite values found in X_test!"
print("All feature values are 100% finite (0 NaNs, 0 Infs)!")

## 4. Class Distribution & Imbalance Preservation Audit
Verify that the extracted label distributions exactly match the original NASA splits.

In [ ]:
from collections import Counter
train_counts = Counter(y_train)
val_counts = Counter(y_val)
test_counts = Counter(y_test)

with open(config.CLASS_MAPPING_PATH) as f:
    class_names = {int(p[0]): p[1].strip() for p in [l.strip().split(None, 1) for l in f if l.strip()]}

df_class_check = pd.DataFrame([
    {
        "Class ID": cid,
        "Class Name": class_names[cid],
        "Train": train_counts[cid],
        "Val": val_counts[cid],
        "Test": test_counts[cid],
        "Total": train_counts[cid] + val_counts[cid] + test_counts[cid]
    }
    for cid in config.CLASS_IDS_RANGE
])
df_class_check